# NxM validation: synthetic injections and noise-only traces

This notebook records the automated test result and validates the profiled frequency-domain reconstruction on seeded correlated noise.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))
from nxm import NxMReconstructor

## Automated tests

In [ ]:
completed = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=REPO_ROOT, capture_output=True, text=True, check=False,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
assert completed.returncode == 0

## Seeded model and correlated noise

In [ ]:
RNG = np.random.default_rng(20261023)
N_CHANNELS, N_MODES, N_SAMPLES = 3, 2, 256
sample = np.arange(N_SAMPLES, dtype=float)
means, components = [], []
for channel in range(N_CHANNELS):
    center, width = 105 + 2 * channel, 7 + channel
    u = (sample - center) / width
    mean = np.exp(-0.5 * u**2)
    candidates = [(u**2 - 0.5) * mean, (u**4 - 3 * u**2 + 0.75) * mean]
    nuisance = [mean / np.linalg.norm(mean), np.gradient(mean) / np.linalg.norm(np.gradient(mean))]
    modes = []
    for candidate in candidates:
        mode = candidate.copy()
        for direction in nuisance + modes:
            mode -= np.dot(mode, direction) * direction
        modes.append(mode / np.linalg.norm(mode))
    means.append(mean)
    components.append(modes)
means = np.asarray(means)
components = np.asarray(components)
variances = np.full((N_CHANNELS, N_MODES), 0.04)

frequency = np.arange(N_SAMPLES // 2 + 1)
correlation = np.array([[1.0, 0.35, 0.15], [0.35, 1.0, 0.25], [0.15, 0.25, 1.0]])
color = 1.0 + 4.0 / (1.0 + (frequency / 6.0)**2)
noise_csd = 0.025 * color[:, None, None] * correlation[None, :, :]
reconstructor = NxMReconstructor(means, components, variances, noise_csd)

def draw_noise(n_events):
    spectra = np.empty((n_events, N_CHANNELS, len(frequency)), dtype=complex)
    for index, covariance in enumerate(noise_csd):
        factor = np.linalg.cholesky(covariance)
        if index == 0 or index == len(frequency) - 1:
            standard = RNG.normal(size=(n_events, N_CHANNELS))
        else:
            standard = (RNG.normal(size=(n_events, N_CHANNELS)) + 1j * RNG.normal(size=(n_events, N_CHANNELS))) / np.sqrt(2)
        spectra[:, :, index] = standard @ factor.T
    return np.fft.irfft(spectra, n=N_SAMPLES, axis=-1)

def synthesize(amplitudes, delays, coefficients):
    phase = np.exp(-2j * np.pi * frequency[:, None] * delays[None, :] / N_SAMPLES)
    mean_fft = np.fft.rfft(means, axis=-1).T
    scaled = components * np.sqrt(variances)[:, :, None]
    basis_fft = np.transpose(np.fft.rfft(scaled, axis=-1), (2, 0, 1))
    shape_fft = np.einsum('fjk,jk->fj', basis_fft * phase[:, :, None], coefficients)
    model_fft = amplitudes[None, :] * (mean_fft * phase + shape_fft)
    return np.fft.irfft(model_fft.T, n=N_SAMPLES, axis=-1)

## Noise-only validation

In [ ]:
noise_results = []
for traces in draw_noise(12):
    noise_results.append(reconstructor.fit(
        traces, initialization_bounds=(-6, 6), delay_half_width=2,
        optimizer_options={'maxiter': 120},
    ))
noise_amplitudes = np.stack([result.amplitudes for result in noise_results])
noise_coefficients = np.stack([result.shape_coefficients for result in noise_results])
noise_reduced_residual = np.array([result.residual_objective for result in noise_results]) / (N_CHANNELS * (N_SAMPLES - 1))
print(f'Converged: {np.mean([result.success for result in noise_results]):.1%}')
print(f'Median reduced residual objective: {np.median(noise_reduced_residual):.3f}')
print(f'Largest noise-only PCA pull: {np.max(np.abs(noise_coefficients)):.3f}')

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
axes[0].hist(noise_amplitudes.ravel(), bins=15); axes[0].set_title('Noise-only amplitudes')
axes[1].hist(noise_coefficients.ravel(), bins=15); axes[1].set_title('Noise-only PCA pulls')
axes[2].hist(noise_reduced_residual, bins=10); axes[2].axvline(1, color='k', ls='--'); axes[2].set_title('Reduced residual')
plt.tight_layout(); plt.show()
assert np.all(np.isfinite(noise_amplitudes))
assert np.all(noise_amplitudes >= 0)
assert np.all(np.isfinite(noise_coefficients))

## Simulated pulse injections

In [ ]:
N_INJECTIONS = 16
true_amplitudes = RNG.uniform(0.8, 2.0, size=(N_INJECTIONS, N_CHANNELS))
true_delays = RNG.uniform(-3.0, 3.0, size=(N_INJECTIONS, N_CHANNELS))
true_coefficients = RNG.normal(0.0, 0.4, size=(N_INJECTIONS, N_CHANNELS, N_MODES))
injection_noise = draw_noise(N_INJECTIONS)
fit_results = []
for index in range(N_INJECTIONS):
    traces = synthesize(true_amplitudes[index], true_delays[index], true_coefficients[index]) + injection_noise[index]
    fit_results.append(reconstructor.fit(
        traces, initialization_bounds=(-6, 6), delay_half_width=2,
        optimizer_options={'maxiter': 160},
    ))
fitted_amplitudes = np.stack([result.amplitudes for result in fit_results])
fitted_delays = np.stack([result.delays for result in fit_results])
fitted_coefficients = np.stack([result.shape_coefficients for result in fit_results])
convergence = np.mean([result.success for result in fit_results])
amplitude_bias = np.median((fitted_amplitudes - true_amplitudes) / true_amplitudes)
delay_mae = np.mean(np.abs(fitted_delays - true_delays))
coefficient_rmse = np.sqrt(np.mean((fitted_coefficients - true_coefficients)**2))
print(f'Convergence: {convergence:.1%}')
print(f'Median fractional amplitude bias: {amplitude_bias:.3%}')
print(f'Delay MAE: {delay_mae:.3f} samples')
print(f'PCA coefficient RMSE: {coefficient_rmse:.3f}')
assert convergence >= 0.95
assert abs(amplitude_bias) < 0.05
assert delay_mae < 0.5
assert coefficient_rmse < 0.35

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].scatter(true_amplitudes, fitted_amplitudes, s=18); axes[0].axline((0, 0), slope=1, color='k', ls='--'); axes[0].set(xlabel='True amplitude', ylabel='Fitted amplitude')
axes[1].scatter(true_delays, fitted_delays, s=18); axes[1].axline((0, 0), slope=1, color='k', ls='--'); axes[1].set(xlabel='True delay [samples]', ylabel='Fitted delay [samples]')
axes[2].scatter(true_coefficients, fitted_coefficients, s=18); axes[2].axline((0, 0), slope=1, color='k', ls='--'); axes[2].set(xlabel='True PCA coefficient', ylabel='Fitted coefficient')
plt.tight_layout(); plt.show()